<h1>Unusable due to 99>% Nan in caravan qual lite in aoi </h1>
<h1>Case Study II: Colorado</h1>
<h3>the United States of America, Rocky Mountains</h3>

<h2>Data Used:</h2>

- Flo1K (Barbarossa et al. 2018): streamflow raster   
- Mindat (Von Bargen et al. 2025): collected from the API in this notebook  
- HydroSHEDS (Lehrer et al. 2008): Flow Direction raster    
- HydroBASINS (Lehner and Grill 2013): Basin vectors    
- Mining polygons (Tang and Werner 2023): Mining area vectors   

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import geopandas as gpd
from data_utils import *
from classes import AMDModel
import contextily as cx
import matplotlib.colors as mcolors
new_model_initrun = True



<h1>Configuration</h1>

In [ ]:
time_first = "2014"
time_last = "2015"
region = "USA"
hydrobasins_region_code = "na"
basins_iloc = (16, 17)

<h2>Collect Mindat data </h2>
<p> Mindat data is used to filter if the mining polygons from Tang and Werner 2023, actually have the mineral of interest. For this case study the mineral is set as pyrite using keyword aguments: <b>material_name: "pyrite", mineral_id: 3314, material_strings: "(Fe|S)</b>.</p>

<p>The region of interest is set to "Spain" through the regular argument: region. </p>

<p> The mindat_collector() function relies on the fact that you have access to the mindat API with an API key saved in the repository. The name of the file can be changed through the keyword argument: <b>mindat_api_str</b>.

In [ ]:
mindat_collector(region)

<h2>Flo1K preperation</h2>
<p>The Flo1K dataset is a large (24 GB) dataset, to make use a bit easier the raster is clipped by the area of interest (aoi) defined with the index locations (i:i) of basins in a HydroBASINS shapefile</p>
<p>For this case study the there is only a single basin (shown below) in the south of Spain around the city of Huelva (close to Sevilla)</p>
<p>Note that the file input and output locations can be changed through the use of keyword arguments, one especially of note is the <b>date</b> keyword argument which takes a numpy datetime64 object as input, only the year argument of this Timestamp should be changed, while the month and day should stay at 1, as the Flo1K dataset works on yearly data set to <b>year/01/01</b>. The <b>date</b> kwarg can also be set to a numpy ndarray of datetime64[D] dtype with index [0] is start and index [-1] of end.</p>

In [ ]:
basins = gpd.read_file(f"../data/hybas_{hydrobasins_region_code}_lev04/hybas_{hydrobasins_region_code}_lev04_v1c.shp")
aoi = basins.iloc[basins_iloc[0]: basins_iloc[1]]
ax = aoi.boundary.plot(figsize=(8, 8), edgecolor = "k")
cx.add_basemap(ax, crs = aoi.crs)

In [ ]:
time_array = np.ndarray(2, dtype = "datetime64[D]")
time_array[0] = np.datetime64(f"{time_first}-01-01")
time_array[-1] = np.datetime64(f"{time_last}-01-01")
flo1k_prep(basins_iloc = basins_iloc, date= time_array, output_path=f"../data/flo_CO_", basins_path= f"../data/hybas_{hydrobasins_region_code}_lev04/hybas_{hydrobasins_region_code}_lev04_v1c.shp")

In [ ]:
flo_aoi_date = xr.open_dataset(f"../data/flo_CO_{time_first}-{time_last}.nc")
ax = aoi.boundary.plot(figsize=(8, 8), edgecolor = "k")
flo_aoi_date_plot = flo_aoi_date["qav"].where(flo_aoi_date["qav"] > 0)
flo_aoi_date_plot = flo_aoi_date_plot.isel(time = 0)
norm = mcolors.LogNorm(vmin = flo_aoi_date_plot.min(), vmax = flo_aoi_date_plot.max())
flo_aoi_date_plot.plot.pcolormesh(ax = ax, cmap = "Blues", norm = norm, cbar_kwargs = {"label": "Annual Streamflow m3/s"})
cx.add_basemap(ax, crs = aoi.crs)
plt.title("");

<p>Extra years after 2015 are added to better align with the validation data, this is done by copying the year 2015 and appending it as 2016></p>

In [ ]:
# # Get current last timestamp
# last_time = flo_aoi_date.time.values[-1]

# # Define new times (extending existing)
# new_times = pd.date_range(last_time, periods=10, freq="YE")

# # Reindex with forward fill
# new_ds = flo_aoi_date.reindex(time=new_times, method="ffill")

# flo_aoi_date = xr.merge((new_ds, flo_aoi_date))

<h2>Flow Direction</h2>
<p>The model requires each cell in the raster to have an unique cell ID (ID), the IDs of where the flow goes to next (outID), and if the cell is the most upstream (source). To create this the HydroSHEDS hydir data is used </p>

In [ ]:
hydir_ds = xr.open_dataset(f"../data/hyd_{hydrobasins_region_code}_dir_30s/hyd_{hydrobasins_region_code}_dir_30s.tif")
hydir_ds = hydir_ds.rename({"band_data": "hydir"})
hydir_ds_ids = hydir_IDs(hydir_ds, aoi)

In [ ]:
ax = hydir_ds_ids["source"].plot(cbar_kwargs = {"label": "Source (1:True/0:False)"})
plt.title("")
plt.show()
plt.close()

<h2>Mining Polygons and Mineral Occurences</h2>
<p>Because no good dataset exists where mining areas are matched with what ores are present this matching has to be here. The principle is as follows, the mining polygons (Tang and Werner 2013) are first converted to raster datasets (to align with the other datasets) with a bolean array of mine or no mine. The mindat mineral occurences are points on a map, but all have some error associated with them, thus the mineral points get a buffer of around 5km/0.045 degrees. If a mine cell intersects with the buffer of a mineral occurence the mine is kept, if not the mine likely does not have the mineral and is dropped. </p>

In [ ]:
vector_rasterisation(flo1k_path=f"../data/flo_CO_{time_first}-{time_last}.nc")
mines_raster = xr.open_dataset("../data/mines_raster.tif")
mineral_points = gpd.read_file(f"../data/mindat_data/{region}_pyrite.csv")
mineral_points = gpd.points_from_xy(mineral_points["longitude"], mineral_points["latitude"], crs = "EPSG:4326")
mineral_points = gpd.GeoDataFrame(mineral_points).set_geometry(col = 0)
filtered_mines_raster = filter_mines_with_buffer(mines_raster, mineral_points)

<h2>Merging Datasets</h2>
<p>For the model a single dataset is used, thus the different datasets are merged using xarray.merge.</p>   
<p>Note that labels, descriptions and units are changed in the function cleanup_and_metadata() in the funcs file </p>

In [ ]:
# convert bools to ints
hydir_ds_ids_num = bool_to_int(hydir_ds_ids.copy())   

# reproject
flo_aoi_date = flo_aoi_date.rio.write_crs("EPSG:4326")
ref = flo_aoi_date

hydir_aligned = hydir_ds_ids_num.rio.reproject_match(ref)
mines_aligned = filtered_mines_raster.rio.reproject_match(ref)

# merge
ds_combined = xr.merge([ref, hydir_aligned, mines_aligned])

# cleanup
ds = cleanup_and_metadata(ds_combined)

<h2>Adding Time dimension </h2>
<p>Although the Flo1K dataset already has a time dimension of years 2000-2020, a more discrete timestep is needed for the model, with the function add_time (see funcs.py) this more discrete time dimension is added. For now this only works with a single year, split into more discrete timesteps such as 12 months or 56 weeks, further work is needed to make multiple year splits possible.</p>
<p>Note that the streamflow (Flo1K) is divided by the length of the split, such as Q / 12, to give average timestep streamflow</p>

In [ ]:
ds = add_time(ds, 52, "W")

<h2>Estimating Ore amounts</h2>
<p>For each cell of "mines", a reactive ore amount must be estimated. This is done using the equation:       


$$
reactive_{ore} = 27000f
$$

<p>Wherein (for this test) H (height) is set to 10, and F (fraction of reactive material per m2) is set to 0.01.</p>

In [ ]:
ds = estimate_ore(ds, F =1)

In [ ]:
ds

<h2>Modelling</h2>

In [ ]:
model = AMDModel(ds, "week", output_path= f"../data/validation data/AMDFLOW_CO_{time_first}-{time_last}_W.nc")

In [ ]:
if new_model_initrun:
    model.run(backend = "threading", chunk_size= 1000)